# Assignment 4

In [4]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle

---

## Part 1 - Random Graph Identification

For the first part of this assignment you will analyze randomly generated graphs and determine which algorithm created them.

In [5]:
G1 = nx.read_gpickle("assets/A4_P1_G1")
G2 = nx.read_gpickle("assets/A4_P1_G2")
G3 = nx.read_gpickle("assets/A4_P1_G3")
G4 = nx.read_gpickle("assets/A4_P1_G4")
G5 = nx.read_gpickle("assets/A4_P1_G5")
P1_Graphs = [G1, G2, G3, G4, G5]

AttributeError: module 'networkx' has no attribute 'read_gpickle'

In [11]:
# Since Networkx 3.0 removed read_pickle, in this env. we will use pickle

def read_gpickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


G1 = read_gpickle("assets/A4_P1_G1")
G2 = read_gpickle("assets/A4_P1_G2")
G3 = read_gpickle("assets/A4_P1_G3")
G4 = read_gpickle("assets/A4_P1_G4")
G5 = read_gpickle("assets/A4_P1_G5")
P1_Graphs = [G1, G2, G3, G4, G5]
P1_Graphs

<br>
`P1_Graphs` is a list containing 5 networkx graphs. Each of these graphs were generated by one of three possible algorithms:
* Preferential Attachment (`'PA'`)
* Small World with low probability of rewiring (`'SW_L'`)
* Small World with high probability of rewiring (`'SW_H'`)

Anaylze each of the 5 graphs using any methodology and determine which of the three algorithms generated each graph.

*The `graph_identification` function should return a list of length 5 where each element in the list is either `'PA'`, `'SW_L'`, or `'SW_H'`.*

In [46]:
def graph_identification():
    # YOUR CODE HERE
    graph_algos = []

    def classify_graph(G: nx.Graph):
        degrees = [d for _, d in G.degree()]
        deg_array  = np.array(degrees)

        clustering = nx.average_clustering(G)
        avg_degree = np.mean(deg_array)
        std_degree = np.std(deg_array)
        max_degree = np.max(deg_array)

        ceof_var = std_degree / avg_degree

        if nx.is_connected(G):
            avg_path = nx.average_shortest_path_length(G)
        else:
            lcc = G.subgraph(max(nx.connected_components(G), key=len))
            avg_path = nx.average_shortest_path_length(lcc)

        if ceof_var > 1.0:
            label = "PA"
        elif clustering > 0.3 and avg_path > 4.0:
            label = "SW_L"
        else:
            label = "SW_H"

        return label

    for G in P1_Graphs:
        graph_algos.append(classify_graph(G))

    return graph_algos

In [47]:
ans_one = graph_identification()
assert type(ans_one) == list, "You must return a list"


In [48]:
ans_one

['PA', 'SW_L', 'SW_L', 'PA', 'SW_L']

---

## Part 2 - Company Emails

For the second part of this assignment you will be working with a company's email network where each node corresponds to a person at the company, and each edge indicates that at least one email has been sent between two people.

The network also contains the node attributes `Department` and `ManagmentSalary`.

`Department` indicates the department in the company which the person belongs to, and `ManagmentSalary` indicates whether that person is receiving a managment position salary.

In [22]:
G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))

print(f"Graph with {len(nx.nodes(G))} nodes and {len(nx.edges(G))} edges")

Graph with 1005 nodes and 16706 edges


### Part 2A - Salary Prediction

Using network `G`, identify the people in the network with missing values for the node attribute `ManagementSalary` and predict whether or not these individuals are receiving a managment position salary.

To accomplish this, you will need to create a matrix of node features of your choice using networkx, train a sklearn classifier on nodes that have `ManagementSalary` data, and predict a probability of the node receiving a managment salary for nodes where `ManagementSalary` is missing.



Your predictions will need to be given as the probability that the corresponding employee is receiving a managment position salary.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a Pandas series of length 252 with the data being the probability of receiving managment salary, and the index being the node id.

    Example:
    
        1       1.0
        2       0.0
        5       0.8
        8       1.0
            ...
        996     0.7
        1000    0.5
        1001    0.0
        Length: 252, dtype: float64

In [23]:
list(G.nodes(data=True))[:5] # print the first 5 nodes

[(0, {'Department': 1, 'ManagementSalary': 0.0}),
 (1, {'Department': 1, 'ManagementSalary': nan}),
 (581, {'Department': 3, 'ManagementSalary': 0.0}),
 (6, {'Department': 25, 'ManagementSalary': 1.0}),
 (65, {'Department': 4, 'ManagementSalary': nan})]

In [34]:
def is_management_salary(node):
        ManagementSalary = node[1]["ManagementSalary"]
        if ManagementSalary == 0:
            return 0
        elif ManagementSalary == 1:
            return 1
        else:
            return None
G_df = pd.DataFrame(index=G.nodes())
G_df["degree"] = pd.Series(dict(G.degree()).values())
G_df["degree_centrality"] = pd.Series(nx.degree_centrality(G))
G_df["betweenness_centrality"] = pd.Series(nx.betweenness_centrality(G, normalized=True))
G_df["is_management_salary"] = pd.Series([is_management_salary(node) for node in G.nodes(data=True)])

train = G_df[~pd.isnull(G_df["is_management_salary"])]
test = G_df[pd.isnull(G_df["is_management_salary"])]
features = ["degree", "degree_centrality", "betweenness_centrality"]
X_train = train[features]
y_train = train["is_management_salary"]
X_test = test[features]

X_train

,degree,degree_centrality,betweenness_centrality
0,44,0.043825,0.001124
6,31,0.114542,0.012387
65,39,0.090637,0.012473
64,132,0.169323,0.021924
73,91,0.030876,0.000413
...,...,...,...
862,2,0.000996,0.000000
798,3,0.001992,0.000000
965,1,0.000996,0.000000
973,1,0.000996,0.000000


In [49]:
def salary_predictions():
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    # YOUR CODE HERE
    def build_features(G):
        degree= dict(G.degree())
        clustering= nx.clustering(G)
        betweenness= nx.betweenness_centrality(G)
        closeness= nx.closeness_centrality(G)
        pagerank= nx.pagerank(G)
        avg_nbr_degree= nx.average_neighbor_degree(G)

        mgmt_known = {n: G.nodes[n]['ManagementSalary']
                  for n in G.nodes()
                  if not np.isnan(G.nodes[n].get('ManagementSalary', np.nan))}

        mgmt_nbr_count = {}
        mgmt_nbr_ratio = {}
        for n in G.nodes():
            nbrs = list(G.neighbors(n))
            known_nbrs = [v for v in nbrs if v in mgmt_known]
            mgmt_count = sum(mgmt_known[v] for v in known_nbrs)
            mgmt_nbr_count[n] = mgmt_count
            mgmt_nbr_ratio[n] = mgmt_count / len(nbrs) if nbrs else 0.0

        rows = []
        for n in G.nodes():
            dept = G.nodes[n].get('Department', -1)
            rows.append({
                'node':             n,
                'degree':           degree[n],
                'clustering':       clustering[n],
                'betweenness':      betweenness[n],
                'closeness':        closeness[n],
                'pagerank':         pagerank[n],
                'avg_nbr_degree':   avg_nbr_degree[n],
                'mgmt_nbr_count':   mgmt_nbr_count[n],
                'mgmt_nbr_ratio':   mgmt_nbr_ratio[n],
                'department':       dept,
                'ManagementSalary': G.nodes[n].get('ManagementSalary', np.nan),
            })

        return pd.DataFrame(rows).set_index('node')

    df = build_features(G)
    
    feature_cols = [
    'degree', 'clustering', 'betweenness', 'closeness',
    'pagerank', 'avg_nbr_degree', 'mgmt_nbr_count', 'mgmt_nbr_ratio',
    'department'
    ]

    labeled   = df[df['ManagementSalary'].notna()]
    unlabeled = df[df['ManagementSalary'].isna()]

    X_train = labeled[feature_cols]
    y_train = labeled['ManagementSalary'].astype(int)
    X_test  = unlabeled[feature_cols]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rfc = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=23)
    rfc.fit(X_train_scaled, y_train)
    y_pred = rfc.predict_proba(X_test_scaled)[:, 1]

    return pd.Series(y_pred, index=X_test.index)

In [50]:
ans_salary_preds = salary_predictions()
assert type(ans_salary_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_salary_preds) == 252, "The series must be of length 252"


### Part 2B - New Connections Prediction

For the last part of this assignment, you will predict future connections between employees of the network. The future connections information has been loaded into the variable `future_connections`. The index is a tuple indicating a pair of nodes that currently do not have a connection, and the `Future Connection` column indicates if an edge between those two nodes will exist in the future, where a value of 1.0 indicates a future connection.

In [37]:
future_connections = pd.read_csv('assets/Future_Connections.csv', index_col=0, converters={0: eval})
future_connections.head(10)

,Future Connection
"(6, 840)",0.0
"(4, 197)",0.0
"(620, 979)",0.0
"(519, 872)",0.0
"(382, 423)",0.0
"(97, 226)",1.0
"(349, 905)",0.0
"(429, 860)",0.0
"(309, 989)",0.0
"(468, 880)",0.0


Using network `G` and `future_connections`, identify the edges in `future_connections` with missing values and predict whether or not these edges will have a future connection.

To accomplish this, you will need to:      
1. Create a matrix of features of your choice for the edges found in `future_connections` using Networkx     
2. Train a sklearn classifier on those edges in `future_connections` that have `Future Connection` data     
3. Predict a probability of the edge being a future connection for those edges in `future_connections` where `Future Connection` is missing.



Your predictions will need to be given as the probability of the corresponding edge being a future connection.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a series of length 122112 with the data being the probability of the edge being a future connection, and the index being the edge as represented by a tuple of nodes.

    Example:
    
        (107, 348)    0.35
        (542, 751)    0.40
        (20, 426)     0.55
        (50, 989)     0.35
                  ...
        (939, 940)    0.15
        (555, 905)    0.35
        (75, 101)     0.65
        Length: 122112, dtype: float64

In [40]:
def new_connections_predictions():
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.ensemble import RandomForestClassifier

    # YOUR CODE HERE
    future_connections['preferential_attachment'] = [list(nx.preferential_attachment(G, [node_pair]))[0][2]
                                             for node_pair in future_connections.index]
    future_connections['common_neighbors'] = [len(list(nx.common_neighbors(G, node_pair[0], node_pair[1]))) 
                                            for node_pair in future_connections.index]
    train_data = future_connections[~future_connections['Future Connection'].isnull()]
    test_data = future_connections[future_connections['Future Connection'].isnull()]

    X_train = train_data[["preferential_attachment", "common_neighbors"]].values
    y_train = train_data["Future Connection"].values
    X_test = test_data[["preferential_attachment", "common_neighbors"]].values

    rfc = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=23)
    rfc.fit(X_train, y_train)
    y_pred = rfc.predict_proba(X_test)[:, 1]
    return pd.Series(y_pred, index=test_data.index)

In [41]:
ans_prob_preds = new_connections_predictions()
assert type(ans_prob_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_prob_preds) == 122112, "The series must be of length 122112"
